In [2]:
# Cell 0: Setup with your exact Kaggle paths
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
from torchvision import models, transforms

print("Setup complete")

# === YOUR EXACT PATHS ===
MAIN_ANNO_ROOT   = Path("/kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos")
TRACKING_ROOT    = Path("/kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation")

OUTPUT_BASE = Path("/kaggle/working/processed")
for folder in ["group_labels", "crops", "features", "sequences", "checkpoints"]:
    (OUTPUT_BASE / folder).mkdir(parents=True, exist_ok=True)

print("All folders created!")
print("Main annotations path:", MAIN_ANNO_ROOT)
print("Tracking annotations path:", TRACKING_ROOT)

Setup complete
All folders created!
Main annotations path: /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos
Tracking annotations path: /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation


In [3]:
# Cell 1: Load group labels ONLY from main annotations.txt files
group_labels = {}

# Scan only in the videos folder for annotations.txt
for anno_file in tqdm(list(MAIN_ANNO_ROOT.glob("**/*annotations.txt"))):
    with open(anno_file, "r") as f:
        for line in f:
            if line.strip():
                parts = line.strip().split()
                if len(parts) >= 2:
                    frame_name = parts[0]           
                    group_label = parts[1]          
                    group_labels[frame_name] = group_label

print(f"Loaded {len(group_labels)} group labels")
print("Example:", list(group_labels.items())[:5])


pd.DataFrame(list(group_labels.items()), columns=["frame", "group_label"]).to_csv(
    OUTPUT_BASE / "group_labels/group_labels.csv", index=False
)
print("Group labels saved!")

100%|██████████| 55/55 [00:00<00:00, 149.31it/s]

Loaded 4131 group labels
Example: [('102095.jpg', 'r_spike'), ('114725.jpg', 'l-pass'), ('64555.jpg', 'r-pass'), ('93570.jpg', 'r_spike'), ('62810.jpg', 'r_set')]
Group labels saved!


In [4]:
# Cell 2: Load tracking annotations (only *.txt files like 13286.txt)
processed_count = 0

for video_folder in tqdm(list(TRACKING_ROOT.glob("*"))):          
    for frame_folder in video_folder.glob("*"):                   
        txt_file = frame_folder / f"{frame_folder.name}.txt"
        if not txt_file.exists():
            continue

        frame_name = f"{frame_folder.name}.jpg"
        group_label = group_labels.get(frame_name, "unknown")

      
        base_num = int(frame_folder.name)
        frames = []
        for i in range(base_num - 4, base_num + 5):
            img_path = MAIN_ANNO_ROOT / video_folder.name / frame_folder.name / f"{i:05d}.jpg"
            if img_path.exists():
                img = cv2.imread(str(img_path))
                if img is not None:
                    frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        if len(frames) != 9:
            continue

        
        num_players = len(open(txt_file).readlines())

        print(f"✓ {frame_name} | Group: {group_label} | Frames: 9 | Players: {num_players}")
        processed_count += 1

print(f"\nCell 2 finished! Processed {processed_count} sequences successfully.")

  0%|          | 0/56 [00:00<?, ?it/s]

✓ 96940.jpg | Group: l-spike | Frames: 9 | Players: 240
✓ 64555.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 60550.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 81900.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 96865.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 101975.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 62725.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 51665.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 64385.jpg | Group: l_set | Frames: 9 | Players: 240
✓ 96915.jpg | Group: l_set | Frames: 9 | Players: 240
✓ 102005.jpg | Group: l_set | Frames: 9 | Players: 240
✓ 60570.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 96995.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 81820.jpg | Group: r_set | Frames: 9 | Players: 220
✓ 52740.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 36810.jpg | Group: l_winpoint | Frames: 9 | Players: 220
✓ 96065.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 102095.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 1145

  2%|▏         | 1/56 [00:17<15:56, 17.40s/it]

✓ 62785.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 14170.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 20760.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 20745.jpg | Group: l-spike | Frames: 9 | Players: 240
✓ 35160.jpg | Group: l_winpoint | Frames: 9 | Players: 240
✓ 13095.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 11915.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 13215.jpg | Group: l-pass | Frames: 9 | Players: 200
✓ 12845.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 20800.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 13290.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 39660.jpg | Group: l_set | Frames: 9 | Players: 240
✓ 32805.jpg | Group: l_winpoint | Frames: 9 | Players: 240
✓ 23715.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 22605.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 22710.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 12775.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 13260.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 1

  4%|▎         | 2/56 [00:28<12:17, 13.67s/it]

...

✓ 10675.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 21620.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 24230.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 21690.jpg | Group: l_set | Frames: 9 | Players: 240
✓ 35195.jpg | Group: l-spike | Frames: 9 | Players: 240
✓ 52685.jpg | Group: l-spike | Frames: 9 | Players: 240
✓ 30275.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 30435.jpg | Group: l-spike | Frames: 9 | Players: 240
✓ 17465.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 17245.jpg | Group: l_set | Frames: 9 | Players: 240
✓ 30585.jpg | Group: r_winpoint | Frames: 9 | Players: 240
✓ 30395.jpg | Group: l_set | Frames: 9 | Players: 220
✓ 74455.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 52535.jpg | Group: l-spike | Frames: 9 | Players: 220
✓ 30520.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 17415.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 17385.jpg | Group: r_set | Frames: 9 | Players: 240
✓ 39705.jpg | Group: l_winpoint | Frames: 9 | Players: 240
✓ 171

 98%|█████████▊| 55/56 [12:55<00:11, 11.13s/it]

✓ 21675.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 31755.jpg | Group: l_winpoint | Frames: 9 | Players: 240
✓ 32295.jpg | Group: l_set | Frames: 9 | Players: 240
✓ 12685.jpg | Group: l_set | Frames: 9 | Players: 240
✓ 34200.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 34330.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 37570.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 22655.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 39870.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 34135.jpg | Group: r-pass | Frames: 9 | Players: 200
✓ 39130.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 37545.jpg | Group: r_set | Frames: 9 | Players: 220
✓ 32235.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 39315.jpg | Group: l-pass | Frames: 9 | Players: 240
✓ 34105.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 40115.jpg | Group: r-pass | Frames: 9 | Players: 240
✓ 39910.jpg | Group: l-spike | Frames: 9 | Players: 240
✓ 22545.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 32

100%|██████████| 56/56 [13:08<00:00, 14.09s/it]

✓ 12625.jpg | Group: r_spike | Frames: 9 | Players: 240
✓ 12545.jpg | Group: r-pass | Frames: 9 | Players: 240

Cell 2 finished! Processed 4573 sequences successfully.


In [12]:
# Cell 3 - FULL VERSION (processes ALL sequences)

import torch
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import numpy as np
from torch.cuda.amp import autocast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

resnet = models.resnet18(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet = resnet.to(device)
resnet.eval()

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

feature_dir = OUTPUT_BASE / "features"
feature_dir.mkdir(exist_ok=True)

print("Starting FULL feature extraction on ALL sequences...")

tracking_files = list(TRACKING_ROOT.glob("**/*.txt"))

for txt_file in tqdm(tracking_files):         
    frame_name = txt_file.stem + ".jpg"
    if frame_name not in group_labels:
        continue

    frame_folder = txt_file.parent
    video_folder = frame_folder.parent

    # Load 9 frames
    base_num = int(txt_file.stem)
    frames = []
    for i in range(base_num - 4, base_num + 5):
        img_path = MAIN_ANNO_ROOT / video_folder.name / frame_folder.name / f"{i:05d}.jpg"
        if img_path.exists():
            img = cv2.imread(str(img_path))
            if img is not None:
                frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if len(frames) != 9:
        continue

    # Collect crops
    crops = []
    with open(txt_file, "r") as f:
        for line in f:
            try:
                parts = line.strip().split()
                xmin, ymin, xmax, ymax = map(int, parts[1:5])
                for frame_img in frames:
                    crop = frame_img[ymin:ymax, xmin:xmax]
                    if crop.size > 0:
                        crops.append(transform(crop))
            except:
                continue

    if len(crops) == 0:
        continue

    # Batch inference
    batch_size = 128
    features_list = []
    for i in range(0, len(crops), batch_size):
        batch = torch.stack(crops[i:i+batch_size]).to(device)
        with torch.no_grad(), autocast():
            feat = resnet(batch)
            feat = feat.squeeze(-1).squeeze(-1)
        features_list.append(feat.cpu().numpy())

    seq_features = np.concatenate(features_list, axis=0)
    np.save(feature_dir / f"{txt_file.stem}_features.npy", seq_features)

print("FULL feature extraction completed!")
print(f"Total features saved: {len(list(feature_dir.glob('*.npy')))}")

Device: cuda
Starting FULL feature extraction on ALL sequences...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
  0%|          | 0/4831 [00:00<?, ?it/s]/tmp/ipykernel_55/1587401075.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
100%|██████████| 4831/4831 [6:13:56<00:00,  4.64s/it]  

✅ FULL feature extraction completed!
Total features saved: 3880
